[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/13-permutacao/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/13-permutacao")
    print("Material preparado em:", Path.cwd())


# Testes de permutação em um experimento A/B

Capítulo de estudo das Aulas 13 e 14

# Objetivos

Este notebook usa uma única campanha A/B para construir o teste de
permutação, auditar suas hipóteses e avançar para estatísticas robustas
e permutações estratificadas. Ao final, você deverá saber **o que pode
ser permutado e por quê**.

## Como estudar este capítulo

O teste de permutação constrói a distribuição nula quebrando, de maneira
controlada, a associação entre grupo e resposta. Em um experimento
aleatorizado, a atribuição original dos rótulos fornece a justificativa:
sob a hipótese de ausência de efeito, redistribuições compatíveis com o
desenho são alternativas plausíveis à atribuição observada.

Cada permutação conserva os valores da resposta e reorganiza os rótulos
permitidos. Calculamos a mesma estatística em todas as réplicas e
comparamos o efeito observado com essa distribuição nula. O valor-p é a
fração de estatísticas permutadas tão ou mais extremas que a observada.

As Aulas 13 e 14 compartilham este capítulo. A primeira desenvolve o
caso A/B básico. A segunda pergunta o que muda com estatísticas
assimétricas, estratos, pares ou dependência. A regra central é
preservar o desenho: não se deve permutar livremente observações que só
são trocáveis dentro de blocos ou pares.

# 1. Download da base

In [ ]:
import kagglehub
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")
csv_path = next(Path(path).rglob("marketing_AB.csv"))
df = pd.read_csv(csv_path).drop(columns=["Unnamed: 0"])
df.head()

> **Interpretação**
>
> Antes de interpretar resultados, confirme o que cada linha representa,
> o período coberto e as colunas realmente disponíveis. Essa definição
> determina quais agregações e comparações são válidas.

# 2. O que é a base?

A unidade observacional é um usuário. `test group` registra a atribuição
ao anúncio comercial (`ad`, tratamento) ou PSA (`psa`, controle);
`converted` é a resposta primária. `total ads`, `most ads day` e
`most ads hour` descrevem a exposição registrada durante a campanha.

In [ ]:
print(df.shape)
print(df.isna().sum())
print("IDs duplicados:", df["user id"].duplicated().sum())
df.groupby("test group").agg(
    n=("converted", "size"),
    conversoes=("converted", "sum"),
    taxa=("converted", "mean"),
    media_exposicoes=("total ads", "mean"),
    mediana_exposicoes=("total ads", "median"),
)

# 3. Análise descritiva

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["test group"].value_counts().plot.bar(ax=axes[0])
df.groupby("test group")["converted"].mean().mul(100).plot.bar(ax=axes[1])
axes[0].set_title("Tamanho dos grupos")
axes[1].set_title("Conversão observada")
axes[1].set_ylabel("Conversão (%)")
plt.show()

> **Interpretação**
>
> A permutação constrói a referência de H0 ao quebrar a associação entre
> rótulo e resposta. Isso exige trocaabilidade; blocos, pares ou
> clusters do desenho original precisam ser preservados.

O forte desbalanceamento não prova falha de randomização, mas reduz a
precisão do controle. A documentação pública afirma que se trata de um
A/B, porém não detalha o algoritmo de atribuição; registre essa
limitação.

# 4. Efeito observado

In [ ]:
rates = df.groupby("test group")["converted"].mean()
observed = rates["ad"] - rates["psa"]
relative = rates["ad"] / rates["psa"] - 1
print(f"Diferença absoluta: {observed:.4%}")
print(f"Uplift relativo: {relative:.1%}")

> **Interpretação**
>
> A diferença absoluta informa quantas conversões adicionais ocorreram
> por usuário exposto; o uplift relativo compara essa diferença com a
> taxa do controle. São escalas complementares e ambas devem acompanhar
> a incerteza.

# 5. Uma função geral de permutação

In [ ]:
rng = np.random.default_rng(1213)

def difference_in_means(y, group):
    return y[group == "ad"].mean() - y[group == "psa"].mean()

def permutation_test(y, group, statistic=difference_in_means,
                     B=5000, alternative="two-sided", seed=1213):
    local_rng = np.random.default_rng(seed)
    observed = statistic(y, group)
    null = np.array([
        statistic(y, local_rng.permutation(group))
        for _ in range(B)
    ])
    if alternative == "greater":
        extreme = null >= observed
    elif alternative == "less":
        extreme = null <= observed
    else:
        extreme = np.abs(null) >= abs(observed)
    p_value = (extreme.sum() + 1) / (B + 1)
    return observed, null, p_value

# 6. Teste da conversão

Para execução rápida em computadores modestos, retire uma amostra
estratificada. Depois compare com a versão eficiente da seção seguinte.

In [ ]:
small = df.groupby("test group", group_keys=False).sample(n=10000, random_state=12)
y = small["converted"].astype(float).to_numpy()
g = small["test group"].to_numpy()
t_obs, null, p_value = permutation_test(y, g, B=5000)
print(t_obs, p_value)

sns.histplot(null, bins=40)
plt.axvline(t_obs, color="darkorange", linewidth=3)
plt.xlabel("Diferença de conversão sob H0")
plt.show()

> **Interpretação**
>
> A distribuição permutada reúne diferenças produzidas quando o rótulo
> de grupo não carrega efeito. O valor-p é a fração de permutações pelo
> menos tão extremas quanto a diferença observada.

# 7. Atalho exato para uma resposta binária

Condicionado ao número total de conversões e aos tamanhos dos grupos, o
número de conversões atribuídas a `ad` segue uma distribuição
hipergeométrica.

In [ ]:
n_ad = (df["test group"] == "ad").sum()
n_total = len(df)
n_conv = df["converted"].sum()
B = 100_000
conv_ad = rng.hypergeometric(n_conv, n_total-n_conv, n_ad, size=B)
null_fast = conv_ad/n_ad - (n_conv-conv_ad)/(n_total-n_ad)
p_fast = (1 + np.sum(np.abs(null_fast) >= abs(observed))) / (B+1)
print(p_fast)

> **Interpretação**
>
> Para uma resposta binária, o atalho hipergeométrico reproduz a lógica
> da permutação condicionando o total de conversões. Ele evita
> embaralhar milhões de rótulos sem mudar a hipótese nula.

# 8. Uma ou duas caudas

In [ ]:
p_greater = (1 + np.sum(null_fast >= observed)) / (B+1)
p_two = (1 + np.sum(np.abs(null_fast) >= abs(observed))) / (B+1)
print({"unilateral": p_greater, "bilateral": p_two})

Defina a direção antes de olhar os resultados. Não selecione unilateral
apenas porque o efeito observado ficou na direção desejada.

# Parte II — Aula 14: permutação além do caso básico

A Aula 13 tratou do cenário mais simples: dois grupos, unidades
independentes e rótulos livremente permutáveis. Esse caso é a porta de
entrada, não uma receita universal. Na prática, três escolhas precisam
ser justificadas antes de executar o teste:

1.  **qual estatística representa a pergunta científica?**
2.  **quais unidades podem realmente trocar de rótulo?**
3.  **qual hipótese nula a simulação representa?**

Nesta segunda parte, cada extensão nasce de uma alteração em uma dessas
escolhas. A estatística pode mudar porque média, mediana e média aparada
descrevem aspectos diferentes. O conjunto de permutações pode diminuir
porque o desenho possui estratos, pares ou clusters. A interpretação
também pode mudar conforme testamos ausência de qualquer efeito
individual, efeito médio zero ou igualdade de distribuições.

> **Regra de leitura**
>
> Um teste de permutação não é definido apenas pela linha de código que
> embaralha os rótulos. Ele é definido pelo trio **hipótese nula +
> estatística + conjunto de permutações admissíveis**.

# 9. Uma métrica assimétrica

A conversão é binária, mas `total ads` é uma contagem com forte
assimetria à direita: muitos usuários veem poucos anúncios e poucos
usuários acumulam muitas exposições. Nessa situação, resumir os grupos
por um único número exige decidir qual aspecto da distribuição
interessa.

- A **média** responde pela exposição total dividida entre os usuários e
  é sensível aos casos muito expostos.
- A **mediana** descreve uma posição central: metade dos usuários fica
  abaixo e metade acima.
- A **média aparada** remove uma proporção previamente definida das duas
  caudas e calcula a média do restante.

Esses resumos não são versões intercambiáveis da mesma pergunta. Se um
anúncio aumentar apenas a cauda superior, a média poderá mudar mesmo
quando a mediana permanecer praticamente constante.

In [ ]:
cap = df["total ads"].quantile(.995)
exposure = df[df["total ads"] <= cap].copy()
sns.boxplot(data=exposure, x="test group", y="total ads", showfliers=False)
plt.show()

exposure.groupby("test group")["total ads"].agg(["mean", "median", "std"])

> **Interpretação**
>
> A média reage fortemente à cauda de usuários muito expostos, enquanto
> a mediana descreve um usuário central. Como exposição pode ocorrer
> depois da atribuição, este contraste é descritivo e não deve ser
> tratado automaticamente como efeito causal.

Compare diferenças de média, mediana e média aparada. Explique qual
aspecto de comportamento cada estatística representa. Lembre que
exposição pode ser uma variável pós-tratamento: esta análise é
descritiva, não um ajuste causal.

# 10. Estatísticas robustas

Para dois grupos, podemos usar como estatística qualquer função que seja
recalculada exatamente da mesma maneira após cada permutação. Três
escolhas são

$$
\begin{aligned}
T_{\text{média}} &= \bar X_{ad}-\bar X_{psa},\\
T_{\text{mediana}} &= \widetilde X_{ad}-\widetilde X_{psa},\\
T_{\text{aparada}} &= \bar X_{ad,\alpha}-\bar X_{psa,\alpha}.
\end{aligned}
$$

Aqui, $\bar X_g$ é a média do grupo $g$, $\widetilde X_g$ é sua mediana
e $\bar X_{g,\alpha}$ é a média após remover a fração $\alpha$ de cada
cauda. Nesta aula usamos $\alpha=0{,}10$.

## Um exemplo pequeno antes do código

Considere as exposições de cinco usuários:

$$1, 2, 3, 4, 100.$$

A média é $22$, a mediana é $3$ e a média aparada em 20% também é $3$,
pois removemos $1$ e $100$ antes de calcular $(2+3+4)/3$. O exemplo não
demonstra que a média seja inadequada: o valor $100$ pode ser real e
substantivamente importante. Ele mostra que a escolha do resumo
determina quanto a cauda superior participa da comparação.

> **Robustez não significa apagar dados incômodos**
>
> O nível de aparagem deve ser definido por uma justificativa
> substantiva ou pelo plano de análise, não escolhido depois de observar
> qual versão produz o menor valor-p.

In [ ]:
from scipy.stats import trim_mean

def median_difference(y, group):
    return np.median(y[group == "ad"]) - np.median(y[group == "psa"])

def trimmed_difference(y, group):
    return trim_mean(y[group == "ad"], .10) - trim_mean(y[group == "psa"], .10)

sample = exposure.groupby("test group", group_keys=False).sample(n=5000, random_state=13)
y_ads = sample["total ads"].to_numpy()
g_ads = sample["test group"].to_numpy()

for statistic in [difference_in_means, median_difference, trimmed_difference]:
    result = permutation_test(y_ads, g_ads, statistic=statistic, B=2000)
    print(statistic.__name__, result[0], result[2])

> **Interpretação**
>
> Estatísticas diferentes testam aspectos diferentes da distribuição.
> Resultados divergentes indicam que o efeito sobre o centro aritmético,
> o centro ordinal e a parte menos extrema dos dados não é o mesmo.

Um valor-p pequeno para a diferença de médias não implica
automaticamente uma diferença de medianas. O teste não pergunta se “os
grupos são diferentes” em sentido genérico; ele pergunta se a
estatística escolhida seria extrema sob as trocas autorizadas por $H_0$.

# 11. Permutação estratificada

Suponha que a atribuição tenha sido realizada separadamente em cada dia.
Uma unidade observada na segunda-feira poderia ter recebido `ad` ou
`psa` dentro da segunda-feira, mas não poderia trocar de lugar com uma
unidade de domingo. A permutação livre inventaria atribuições que o
experimento nunca poderia gerar.

Chamando os estratos de $s=1,\ldots,S$, uma estatística combinada pode
ser

$$
T=\sum_{s=1}^{S}w_s(\widehat p_{ad,s}-\widehat p_{psa,s}),
\qquad w_s\geq0,
\qquad \sum_{s=1}^{S}w_s=1.
$$

Nessa expressão:

- $\widehat p_{g,s}$ é a conversão observada no grupo $g$ dentro do
  estrato $s$;
- $w_s$ é o peso atribuído ao estrato $s$;
- $T$ é a média ponderada das diferenças internas aos estratos.

Pesos proporcionais ao número de usuários fazem estratos maiores
contribuírem mais. Outros pesos podem ser válidos, mas devem ser
definidos antes da análise e mantidos em todas as permutações.

## Algoritmo da permutação estratificada

Para cada réplica:

1.  mantenha respostas, dias e tamanhos dos grupos fixos;
2.  embaralhe `ad` e `psa` **separadamente dentro de cada dia**;
3.  calcule a diferença de conversão em cada dia;
4.  combine as diferenças usando os mesmos pesos;
5.  compare a estatística observada com a distribuição obtida.

O código abaixo implementa exatamente esse procedimento. Ele só é
adequado se o dia for realmente um estrato pré-tratamento compatível com
o mecanismo de atribuição.

In [ ]:
def stratified_stat(frame, label_col="permuted"):
    pieces = frame.groupby("most ads day").apply(
        lambda x: x.loc[x[label_col] == "ad", "converted"].mean()
                - x.loc[x[label_col] == "psa", "converted"].mean(),
    )
    weights = frame["most ads day"].value_counts(normalize=True)
    return np.sum(pieces * weights[pieces.index])

# Uma subamostra mantém o exemplo executável em computadores pessoais.
work = df[["most ads day", "test group", "converted"]].sample(
    n=min(40_000, len(df)), random_state=1213
).copy()
work["permuted"] = work["test group"]
observed_strat = stratified_stat(work)

null_strat = []
for _ in range(300):
    work["permuted"] = work.groupby("most ads day")["test group"].transform(
        lambda x: rng.permutation(x.to_numpy())
    )
    null_strat.append(stratified_stat(work))

print(f"Efeito estratificado observado: {observed_strat:.5f}")
print(f"Centro da distribuição nula: {np.mean(null_strat):.5f}")
print(f"p-valor aproximado: {(1 + np.sum(np.abs(null_strat) >= abs(observed_strat))) / (len(null_strat) + 1):.4f}")

> **Interpretação**
>
> Restringir as trocas ao mesmo dia preserva diferenças sistemáticas
> entre estratos. A comparação só é válida se, dentro de cada dia, os
> rótulos forem trocáveis segundo o desenho do experimento.

## Exemplo didático: por que preservar os estratos?

Imagine que a conversão seja naturalmente maior no fim de semana e que,
por acaso, o grupo `ad` contenha proporção maior de usuários desses
dias. Uma permutação livre mistura composição do calendário com efeito
do anúncio. Ao permutar dentro de cada dia, comparamos tratamento e
controle sob o mesmo contexto temporal e depois agregamos essas
comparações.

Isso não significa que estratificar por qualquer variável observada
melhora o teste. Estratificar por uma variável pós-tratamento pode
condicionar a análise em algo causado pelo próprio tratamento e alterar
a pergunta causal.

# 12. Dados pareados: a troca ocorre dentro de cada par

Em um desenho pareado, cada unidade tratada é associada a uma unidade
controle semelhante, ou a mesma unidade é observada sob duas condições.
O objeto básico deixa de ser cada observação isolada e passa a ser a
diferença do par:

$$D_i=Y_{i,1}-Y_{i,0}.$$

Sob uma nula de ausência de efeito e uma atribuição simétrica dentro do
par, o sinal de $D_i$ pode ser invertido. Uma réplica do teste escolhe
$S_i\in\{-1,+1\}$ e calcula

$$
T_b=\frac{1}{m}\sum_{i=1}^{m}S_{i,b}D_i,
$$

em que $m$ é o número de pares e $S_{i,b}$ é o sinal sorteado para o par
$i$ na réplica $b$. Permutar todas as observações livremente destruiria
o pareamento que controla diferenças entre as unidades.

> **Exemplo**
>
> Se três diferenças forem $D=(2,-1,4)$, uma réplica possível usa sinais
> $S=(-1,+1,-1)$ e produz $(-2-1-4)/3=-7/3$. Não reorganizamos os seis
> resultados individualmente: apenas trocamos qual condição ocupa cada
> posição dentro do par.

# 13. Experimentos em clusters

Se escolas, cidades ou turmas inteiras recebem o tratamento, a
randomização ocorre no nível do cluster. Alunos da mesma escola
compartilham contexto e não foram atribuídos independentemente.
Portanto, os rótulos devem ser permutados entre escolas, mantendo todos
os alunos de uma escola juntos.

Permutar alunos produz artificialmente muito mais unidades aleatórias do
que o desenho realmente possui. Isso costuma estreitar a distribuição
nula e gerar valores-p excessivamente pequenos. Ter milhares de
indivíduos não substitui ter um número adequado de clusters
independentes.

# 14. Qual hipótese nula está sendo testada?

A nula estrita de Fisher afirma ausência de efeito para toda unidade:

$$
H_0^F:Y_i(1)=Y_i(0)\quad\text{para todo }i.
$$

$Y_i(1)$ e $Y_i(0)$ são os resultados potenciais da unidade $i$ sob
tratamento e controle. Sob essa hipótese, o resultado que observaríamos
não muda com a atribuição; por isso, podemos reconstruir exatamente o
resultado sob qualquer randomização permitida.

Uma hipótese de efeito médio zero é mais fraca:

$$
H_0^m:\frac1n\sum_{i=1}^{n}[Y_i(1)-Y_i(0)]=0.
$$

Ela permite efeitos individuais diferentes, positivos e negativos, que
se cancelam. Rejeitar a nula estrita não é logicamente igual a demonstrar
que o efeito médio seja diferente de zero. Para hipóteses médias,
estatísticas studentizadas e aproximações assintóticas costumam ser mais
adequadas.

# 15. Studentização

Quando os grupos possuem tamanhos ou variâncias diferentes, a diferença
bruta de médias pode não ter a mesma escala entre permutações. Uma
alternativa é dividir o efeito por seu erro padrão estimado:

$$
T=\frac{\bar X_{ad}-\bar X_{psa}}
{\sqrt{s_{ad}^2/n_{ad}+s_{psa}^2/n_{psa}}}.
$$

Aqui, $\bar X_g$, $s_g^2$ e $n_g$ são, respectivamente, a média, a
variância amostral e o tamanho do grupo $g$. O numerador mede o efeito
observado; o denominador o coloca na escala de sua incerteza. Em cada
permutação, numerador e denominador precisam ser recalculados.

# 16. Subgrupos e multiplicidade

Depois de observar um efeito médio, é tentador testar cada dia, horário,
região ou perfil de usuário. Se realizarmos muitos testes sob hipóteses
nulas verdadeiras, a chance de encontrar ao menos um resultado
aparentemente significativo cresce.

Para uma família de $K$ testes independentes conduzidos com nível
$\alpha$, a probabilidade de ao menos um falso positivo é

$$1-(1-\alpha)^K.$$

Com $K=20$ e $\alpha=0{,}05$, esse valor é aproximadamente $64\%$. A
Aula 15 desenvolve Bonferroni, Holm e FDR. Aqui, a lição é definir
previamente a família de hipóteses e distinguir análise confirmatória de
exploração.

Um teste máximo por permutação pode guardar, em cada réplica, a maior
estatística entre todos os subgrupos. A referência deixa de ser a
distribuição de um teste isolado e passa a representar a maior surpresa
que a busca completa poderia produzir sob $H_0$.

# 17. Regra de parada e monitoramento

Calcular o valor-p repetidamente e encerrar o experimento assim que ele
cruza $0{,}05$ aumenta a taxa de falso positivo. Cada nova inspeção cria
outra chance de parar em uma flutuação favorável. As alternativas são
definir previamente o tamanho da amostra ou usar métodos sequenciais
planejados para esse monitoramento.

# 18. Poder por simulação

In [ ]:
def power_for_effect(p0=.018, effect=.008, n_each=5000, reps=3000):
    control = rng.binomial(n_each, p0, size=reps) / n_each
    treatment = rng.binomial(n_each, p0+effect, size=reps) / n_each
    se0 = np.sqrt(2*p0*(1-p0)/n_each)
    return np.mean(np.abs(treatment-control) > 1.96*se0)

for n in [1000, 2500, 5000, 10000, 25000]:
    print(n, power_for_effect(n_each=n))

> **Interpretação**
>
> O poder cresce com o tamanho dos grupos porque o mesmo efeito se torna
> maior em relação ao erro amostral. A curva vale para a taxa e o efeito
> especificados; outra alternativa exige novo planejamento.

# 19. Checklist de validade

Antes de interpretar causalmente:

1.  confirme a unidade e a probabilidade de randomização;
2.  verifique duplicatas, perdas e desvio da alocação planejada;
3.  preserve blocos, pares ou clusters na permutação;
4.  não ajuste ingenuamente por variáveis pós-tratamento;
5.  defina métrica primária, cauda e regra de parada;
6.  controle multiplicidade em subgrupos.

# 20. Exercícios

1.  Refaça o teste usando risco relativo em vez de diferença absoluta.
2.  Compare o resultado com um teste-z de duas proporções.
3.  Estime a diferença de conversão em cada dia e teste uma interação.
4.  Simule um experimento em clusters e mostre por que permutar pessoas
    é errado.
5.  Crie um cenário em que média e mediana levem a conclusões
    diferentes.
6.  Explique a diferença entre a nula estrita de Fisher e efeito médio
    zero.

# Fonte e limites

Marketing A/B Testing, Kaggle, `faviovaz/marketing-ab-testing`, licença
CC0. A base é adequada para ensino, mas a documentação disponível não
informa todos os detalhes operacionais da randomização. Essa ausência
deve acompanhar qualquer afirmação causal.

# Guia teórico consolidado das Aulas 13 e 14

## O princípio da permutação

Sob uma hipótese nula de ausência de associação entre tratamento e
resultado, os rótulos podem ser trocáveis. Mantemos os resultados fixos,
permutamos os rótulos e recalculamos a estatística. Isso constrói a
distribuição nula induzida pelo desenho.

Em Monte Carlo, uma estimativa segura do valor-p é

$$\widehat p=\frac{1+\#\{T_b\text{ tão extremo quanto }T_{obs}\}}{B+1},$$

em que:

- $\widehat p$ é o valor-p estimado por simulação;
- $T_{obs}$ é o valor da estatística calculada nos grupos originalmente
  observados;
- $T_b$ é a estatística recalculada na $b$-ésima permutação dos rótulos,
  com $b=1,\ldots,B$;
- $B$ é o número total de permutações aleatórias realizadas;
- $\#\{\cdot\}$ significa “número de ocorrências”: contamos quantas
  estatísticas permutadas são pelo menos tão extremas quanto $T_{obs}$
  sob a alternativa escolhida.

“Tão extremo” depende da hipótese alternativa. Em um teste unilateral à
direita, contamos $T_b\geq T_{obs}$; à esquerda, $T_b\leq T_{obs}$; em
um teste bilateral com uma estatística centrada em zero, é comum contar
$|T_b|\geq|T_{obs}|$.

Os termos $+1$ no numerador e no denominador incluem conceitualmente a
configuração observada entre as possibilidades consideradas. Essa
correção evita reportar valor-p igual a zero quando nenhuma das $B$
permutações aleatórias supera $T_{obs}$. A direção unilateral ou
bilateral deve ser definida antes de observar os resultados.

## Permutação não é bootstrap

Permutação quebra a associação para representar $H_0$. Bootstrap
reamostra unidades para aproximar a incerteza sob uma distribuição
empírica. Eles podem usar código parecido, mas respondem perguntas
diferentes.

## Mapa de decisão para a Aula 14

| Situação do estudo | O que deve ser preservado? | Seção para revisar |
|------------------------|------------------------|------------------------|
| resposta assimétrica ou com extremos | estatística definida antes da análise | 9–10 |
| atribuição feita dentro de estratos | totais de tratamento dentro de cada estrato | 11 |
| observações pareadas | vínculo e troca interna de cada par | 12 |
| tratamento sorteado para clusters | unidades do mesmo cluster juntas | 13 |
| interesse em efeito médio, não nula estrita | escala da estatística e hipótese correta | 14–15 |
| muitos subgrupos ou desfechos | família de testes e regra de busca | 16 |
| monitoramento durante a coleta | regra de parada planejada | 17 |

Esse mapa não substitui o desenho do estudo. Ele serve para localizar a
extensão adequada e verificar se o código imita as atribuições que
realmente poderiam ter ocorrido.

# Questões de revisão das Aulas 13 e 14

1.  O que permanece fixo e o que varia em uma permutação simples de dois
    grupos?
2.  Quando a alternativa deve ser bilateral?
3.  Como permutar quando a atribuição ocorreu separadamente dentro de
    cada estrato?
4.  Qual é a diferença entre a nula estrita de Fisher e uma nula de efeito
    médio zero?
5.  Por que média e mediana podem produzir valores-p diferentes na mesma
    base?
6.  Qual é o erro de permutar indivíduos quando o tratamento foi
    sorteado por escola?
7.  Em um estudo pareado, qual objeto recebe a troca aleatória?
8.  O que a studentização acrescenta à diferença bruta de médias?
9.  Por que escolher a porcentagem de aparagem depois de ver os
    valores-p é problemático?
10. Como inspeções repetidas do resultado alteram a taxa de falso
    positivo?

Respostas comentadas

1.  Resultados e tamanhos dos grupos permanecem fixos; os rótulos variam
    e a estatística é recalculada.
2.  Quando diferenças nas duas direções contradizem $H_0$ e são
    relevantes para a pergunta definida antes dos dados.
3.  Os rótulos devem ser permutados somente dentro de cada estrato,
    reproduzindo o mecanismo de atribuição.
4.  A nula estrita exige $Y_i(1)=Y_i(0)$ para toda unidade; efeito médio
    zero permite efeitos individuais que se cancelam.
5.  Porque as estatísticas descrevem aspectos diferentes da distribuição
    e possuem distribuições nulas próprias.
6.  O procedimento finge que indivíduos foram randomizados
    independentemente, aumenta artificialmente o número de unidades
    aleatórias e tende a subestimar a incerteza.
7.  A orientação tratamento–controle dentro de cada par;
    equivalentemente, sorteamos o sinal da diferença de cada par.
8.  Ela divide o efeito por uma estimativa de sua incerteza, produzindo
    uma escala mais comparável quando tamanhos ou variâncias diferem.
9.  Porque transforma uma escolha metodológica em busca pelo resultado
    mais favorável e aumenta o erro do procedimento completo.
10. Cada inspeção cria uma nova oportunidade de parar em uma flutuação
    extrema; sem método sequencial, o nível nominal deixa de representar
    o erro total.